# Pipeline Hyperparameter Tuning (Parallel)

This notebook performs hyperparameter tuning on the **entire pipeline** using **parallel processing**.
It iterates through different parameter combinations, runs the full pipeline for specified test periods in parallel, and saves all results.

**Test Periods:**
- 2025-09 (Auction) / 2025-09 (Market)
- 2025-10 (Auction) / 2025-10 (Market)

**Output Structure:**
- `/opt/temp/haoyan/param_search/`
    - `metrics/`: Parquet files containing parameters and key metrics.
    - `per_outage/`: Parquet files containing per-outage prediction results.
    - `agg/`: Parquet files containing monthly aggregated prediction results.

In [1]:
from pbase.config.ray import init_ray
import shadow_price_prediction

# Initialize Ray for parallel processing
extra_modules = [shadow_price_prediction]
init_ray(address='ray://10.8.0.36:10001', extra_modules=extra_modules)

import pandas as pd
import numpy as np
import random
from pathlib import Path
from pbase.utils.ray import parallel_equal_pool

# Import tuning utilities for resumable search
from shadow_price_prediction.tuning_utils import (
    load_previous_params,
    sample_params,
    run_single_experiment
)

pd.set_option('display.max_columns', 99)

2025-11-19 16:53:05,914	INFO client_builder.py:242 -- Passing the following kwargs to ray.init() on the server: log_to_driver
SIGTERM handler is not set because current thread is not the main thread.


## 1. Define Parameter Search Space
Define the grid of parameters to sample from.

In [2]:
param_grid = {
    # XGBoost Params (shared between classifier and regressor)
    'clf_xgb_n_estimators': [100, 200, 300, 500],
    'clf_xgb_max_depth': [3, 4, 5, 6],
    'clf_xgb_learning_rate': [0.05, 0.1, 0.2],
    'clf_xgb_gamma': [0, 0.1, 0.2],
    'clf_xgb_min_child_weight': [1, 5, 10],
    'clf_xgb_reg_alpha': [0, 0.1, 1.0],
    'clf_xgb_reg_lambda': [1, 10, 100],

    'reg_xgb_n_estimators': [100, 200, 300, 500],
    'reg_xgb_max_depth': [3, 4, 5, 6],
    'reg_xgb_learning_rate': [0.05, 0.1, 0.2],
    'reg_xgb_gamma': [0, 0.1, 0.2],
    'reg_xgb_min_child_weight': [1, 5, 10],
    'reg_xgb_reg_alpha': [0, 0.1, 1.0],
    'reg_xgb_reg_lambda': [1, 10, 100],
    
    # Logistic Regression Params
    'lr_penalty': ['l2'],
    'lr_C': [0.1, 1.0, 10.0],
    
    # ElasticNet Params
    'enet_alpha': [0.1, 1.0, 10.0],
    'enet_l1_ratio': [0.1, 0.5, 0.9],
    
    # Threshold Config
    'threshold_beta': [0.1, 0.5, 1.0, 2.0, 3.0],
    
    # Ensemble weights
    'clf_xgb_thres': [0.3, 0.5, 0.7],
    'reg_xgb_thres': [0.3, 0.5, 0.7],
}

# Calculate total combinations
total_combinations = 1
for values in param_grid.values():
    total_combinations *= len(values)
    
print(f"Total possible parameter combinations: {total_combinations:,}")

Total possible parameter combinations: 18,366,600,960


# Configuration & Resume Previous Search

In [3]:
N_ITER = 10000  # Total number of experiments to run (including previous ones)
BASE_OUTPUT_DIR = Path('/opt/temp/haoyan/param_search')

# Define Test Periods
TEST_PERIODS = [
    (pd.Timestamp('2025-08-01'), pd.Timestamp('2025-08-01')),
    (pd.Timestamp('2025-09-01'), pd.Timestamp('2025-09-01')),
    (pd.Timestamp('2025-10-01'), pd.Timestamp('2025-10-01')),
]

# Load previously searched parameters
seen_params = load_previous_params(BASE_OUTPUT_DIR)

if seen_params:
    print(f"✓ Resuming: {len(seen_params)} combinations already tested")
    remaining_iterations = N_ITER - len(seen_params)
    
    if remaining_iterations <= 0:
        print(f"\n✓ All {N_ITER} experiments completed!")
        print("  To run more, increase N_ITER.")
    else:
        print(f"  Will run {remaining_iterations} more experiments")
else:
    print("✓ Starting fresh search")
    remaining_iterations = N_ITER

Found 110 previous metric files. Loading parameters...
Loaded 110 previously searched parameter combinations.
✓ Resuming: 110 combinations already tested
  Will run 9890 more experiments


## 3. Prepare Experiments

In [4]:
if remaining_iterations > 0:
    print(f"Preparing {remaining_iterations} new experiments...")
    print("Pre-sampling parameters to ensure uniqueness...")
    
    # PRE-SAMPLE all parameters sequentially to ensure uniqueness
    sampled_params_list = []
    for i in range(remaining_iterations):
        iteration_idx = len(seen_params) + i
        
        # Sample parameters (this updates seen_params)
        params = sample_params(param_grid, seen_params)
        sampled_params_list.append((iteration_idx, params))
    
    print(f"✓ Sampled {len(sampled_params_list)} unique parameter combinations")
    
    # Now prepare experiment tuples with PRE-SAMPLED parameters
    experiment_params = []
    for iteration_idx, params in sampled_params_list:
        experiment_params.append({
            'iteration': iteration_idx,
            'params': params,
            'test_periods': TEST_PERIODS,
            'save_dir': BASE_OUTPUT_DIR
        })
    
    print(f"✓ Prepared {len(experiment_params)} experiments")
    print(f"  Output: {BASE_OUTPUT_DIR}")

Preparing 9890 new experiments...
Pre-sampling parameters to ensure uniqueness...
✓ Sampled 9890 unique parameter combinations
✓ Prepared 9890 experiments
  Output: /opt/temp/haoyan/param_search


## 4. Run Parallel Execution

In [5]:
n_jobs = 12

if remaining_iterations > 0:
    print(f"Starting parallel execution with {min(remaining_iterations, n_jobs)} jobs...")
    
    results = parallel_equal_pool(
        func=run_single_experiment,
        param_dict_list=experiment_params,
        n_jobs=min(remaining_iterations, n_jobs),
        param_serialization='default',
        raise_error=False,
        use_tqdm=True
    )
    
    # Summary
    success_count = sum(1 for r in results if r['status'] == 'success')
    error_count = sum(1 for r in results if r['status'] == 'failed')
    
    print(f"\nExecution Complete!")
    print(f"  Successful: {success_count}")
    print(f"  Failed: {error_count}")
    print(f"  Total completed: {len(seen_params) + success_count} / {N_ITER}")
    
    if error_count > 0:
        print("\nErrors:")
        for r in results:
            if r['status'] == 'failed':
                print(f"  Iteration {r['iteration']}: {r.get('error', 'Unknown')}")

Starting parallel execution with 12 jobs...


  0%|          | 0/9890 [00:00<?, ?it/s]

(PoolActor pid=332211, ip=10.42.8.15) 25-11-19 16:53:26 EST | INFO    | dotenv.main          | _get_stream         | #61  | python-dotenv could not find configuration file .env.
(PoolActor pid=332211, ip=10.42.8.15) 25-11-19 16:53:26 EST | WARNING | pbase.utils.utils    | load_env            | #478 | .env could not be found...
(PoolActor pid=332211, ip=10.42.8.15) 25-11-19 16:53:26 EST | INFO    | pbase.config.base    | load_global_config  | #435 | Loading config from local cached json file...
(PoolActor pid=332211, ip=10.42.8.15) 25-11-19 16:53:26 EST | INFO    | pbase.config.base    | __init__            | #44  | Init global config...
(PoolActor pid=332210, ip=10.42.8.15) 25-11-19 16:53:26 EST | INFO    | dotenv.main          | _get_stream         | #61  | python-dotenv could not find configuration file .env.
(PoolActor pid=332210, ip=10.42.8.15) 25-11-19 16:53:26 EST | WARNING | pbase.utils.utils    | load_env            | #478 | .env could not be found...
(PoolActor pid=332210, ip=

KeyboardInterrupt: 

## 5. Analyze Best Results

In [6]:
metrics_dir = BASE_OUTPUT_DIR / 'metrics'

if metrics_dir.exists():
    all_metrics = []
    for file in metrics_dir.glob('iter_*.parquet'):
        try:
            df = pd.read_parquet(file)
            all_metrics.append(df)
        except Exception as e:
            print(f"Warning: Could not load {file.name}: {e}")
    
    if all_metrics:
        combined_metrics = pd.concat(all_metrics, ignore_index=True)
        
        print(f"Total runs analyzed: {len(combined_metrics)}")
        
        # Best by F1 Score
        if 'F1' in combined_metrics.columns:
            best_idx = combined_metrics['F1'].idxmax()
            best_result = combined_metrics.iloc[best_idx]
            
            print("\n" + "=" * 60)
            print("Best Run (by F1 Score)")
            print("=" * 60)
            print(f"Iteration: {best_result.get('iteration', 'N/A')}")
            print(f"F1 Score: {best_result['F1']:.4f}")
            if 'MAE' in best_result:
                print(f"MAE: {best_result['MAE']:.4f}")
            if 'Precision' in best_result:
                print(f"Precision: {best_result['Precision']:.4f}")
            if 'Recall' in best_result:
                print(f"Recall: {best_result['Recall']:.4f}")
            
            # Print parameters
            print("\nParameters:")
            param_cols = [col for col in combined_metrics.columns 
                         if col not in ['F1', 'Precision', 'Recall', 'MAE', 'RMSE', 'R2',
                                       'iteration', 'timestamp', 'run_id', 'metrics_file',
                                       'per_outage_file', 'agg_file']]
            for col in param_cols:
                print(f"  {col}: {best_result[col]}")
        
        # Best by MAE (lower is better)
        if 'MAE' in combined_metrics.columns:
            best_idx = combined_metrics['MAE'].idxmin()
            best_result = combined_metrics.iloc[best_idx]
            
            print("\n" + "=" * 60)
            print("Best Run (by MAE)")
            print("=" * 60)
            print(f"Iteration: {best_result.get('iteration', 'N/A')}")
            print(f"MAE: {best_result['MAE']:.4f}")
            if 'F1' in best_result:
                print(f"F1 Score: {best_result['F1']:.4f}")
            if 'RMSE' in best_result:
                print(f"RMSE: {best_result['RMSE']:.4f}")
            
            # Print parameters
            print("\nParameters:")
            for col in param_cols:
                print(f"  {col}: {best_result[col]}")

Total runs analyzed: 3


## 6. View Results Summary

In [9]:
if 'combined_metrics' in locals():
    # Display summary statistics
    print("Summary Statistics:")
    print("=" * 60)
    
    metric_cols = ['f1_score', 'precision', 'recall', 'mae', 'rmse']
    available_metrics = [col for col in metric_cols if col in combined_metrics.columns]
    
    summary = combined_metrics[available_metrics].describe()
    print(summary)
    
    # Optionally display the top 5 runs by F1
    if 'F1' in combined_metrics.columns:
        print("\n" + "=" * 60)
        print("Top 5 Runs by F1 Score")
        print("=" * 60)
        
        display_cols = ['iteration'] + available_metrics
        top5 = combined_metrics.nlargest(5, 'F1')[display_cols]
        print(top5.to_string(index=False))

Summary Statistics:
       f1_score  precision    recall         mae         rmse
count  3.000000   3.000000  3.000000    3.000000     3.000000
mean   0.316343   0.314473  0.320927  344.475091  1910.261852
std    0.048320   0.023213  0.073790    6.542887    10.179312
min    0.267681   0.291769  0.247267  336.960047  1899.808735
25%    0.292358   0.302628  0.283967  342.260002  1905.321146
50%    0.317036   0.313486  0.320666  347.559956  1910.833558
75%    0.340674   0.325825  0.357756  348.232612  1915.488411
max    0.364313   0.338163  0.394846  348.905269  1920.143264


In [ ]:
# import shutil

# shutil.rmtree(BASE_OUTPUT_DIR)